# Notebook 04: PaddleOCR Model Fine-Tuning and Evaluation on OFPHIL Supporting Documents
**Phase:** 05 Document Processing & OCR  
**Purpose:** Fine-tune and evaluate the PaddleOCR text recognition architecture (PP-OCRv6) using domain-specific OFPHIL clinical and charity supporting document annotations.  
**Advisory Scope:** Establish PaddleOCR as the primary unified OCR experiment, benchmarked against baseline pretrained PaddleOCR, EasyOCR, and TrOCR.

## Section 1: Environment Diagnostics and Model Configuration
Verifies PaddlePaddle CUDA availability, PaddleOCR PP-OCRv6 models, and hardware runtime.

In [1]:
# Step 1: Dynamic DLL Search Path Setup & Preload for Windows
import os
import sys

# Ensure Windows Jupyter kernel finds torch DLL dependencies (shm.dll, c10.dll, etc.)
torch_lib = os.path.join(sys.prefix, 'Lib', 'site-packages', 'torch', 'lib')
if os.path.exists(torch_lib):
    if hasattr(os, 'add_dll_directory'):
        os.add_dll_directory(torch_lib)
    os.environ['PATH'] = torch_lib + os.pathsep + os.environ.get('PATH', '')

import torch
import json
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import paddle

warnings.filterwarnings('ignore')
os.environ['GLOG_minloglevel'] = '3'
os.environ['PADDLE_PDX_LOGGING_LEVEL'] = 'ERROR'

# Locate Project Root
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project Root: {PROJECT_ROOT}")
print(f"Paddle Version: {paddle.__version__} | Compiled with CUDA: {paddle.is_compiled_with_cuda()}")
if paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0:
    print(f"CUDA Device: {paddle.device.cuda.get_device_name(0)}")
else:
    print("Execution Device: CPU")

c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\venv312\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Project Root: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification
Paddle Version: 3.3.1 | Compiled with CUDA: True
CUDA Device: NVIDIA GeForce RTX 3050 Laptop GPU


## Section 2: Load OFPHIL Document Splits and Annotations
Inspects the train, validation, and untouched test manifests. Preserves patient-level clustering.

In [2]:
SPLIT_DIR = PROJECT_ROOT / 'data' / 'interim' / 'document_processing' / 'splits'
TRAIN_CSV = SPLIT_DIR / 'train_manifest_v2.csv'
VAL_CSV = SPLIT_DIR / 'val_manifest_v2.csv'
TEST_CSV = SPLIT_DIR / 'test_manifest_v2.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_val = pd.read_csv(VAL_CSV)
df_test = pd.read_csv(TEST_CSV)

print(f"[OK] Training Manifest Crops   : {len(df_train):,}")
print(f"[OK] Validation Manifest Crops : {len(df_val):,}")
print(f"[OK] Test Manifest Crops (Held-Out) : {len(df_test):,}")
display(df_train[['crop_id', 'document_type', 'ground_truth_text']].head(5))

[OK] Training Manifest Crops   : 15,043
[OK] Validation Manifest Crops : 3,536
[OK] Test Manifest Crops (Held-Out) : 2,778


,crop_id,document_type,ground_truth_text
0,DOC_001DC8F4_BOX_1.0,assessment,- FINANCIAL ASSISTANCE REFERRAL FORM -
1,DOC_001DC8F4_BOX_10.0,assessment,DR. FLAMINIANO
2,DOC_001DC8F4_BOX_12.0,assessment,Anna Bonoa
3,DOC_001DC8F4_BOX_15.0,assessment,DR. PALIZA
4,DOC_001DC8F4_BOX_2.0,assessment,DE BELEN


## Section 3: Prepare RecData Formats for PaddleOCR
Converts bounding box crops and transcriptions into the standardized PaddleOCR recognition format (`rec_gt_train.txt`, `rec_gt_val.txt`).

In [3]:
DATASET_DIR = PROJECT_ROOT / 'data' / 'interim' / 'document_processing' / 'paddleocr_dataset'
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Create clean label text files (tab-separated image path and ground truth label)
def create_paddle_rec_manifest(df, out_path):
    lines = []
    for _, r in df.iterrows():
        txt = str(r['ground_truth_text']).strip()
        if txt and txt != 'nan':
            lines.append(f"{r['image_filename']}\t{txt}\n")
    with open(out_path, 'w', encoding='utf-8') as f:
        f.writelines(lines)
    print(f"Wrote {len(lines):,} lines to {out_path.name}")

create_paddle_rec_manifest(df_train, DATASET_DIR / 'rec_gt_train.txt')
create_paddle_rec_manifest(df_val, DATASET_DIR / 'rec_gt_val.txt')
create_paddle_rec_manifest(df_test, DATASET_DIR / 'rec_gt_test.txt')

Wrote 15,043 lines to rec_gt_train.txt
Wrote 3,536 lines to rec_gt_val.txt
Wrote 2,778 lines to rec_gt_test.txt


## Section 4: Levenshtein Distance & Evaluation Metric Calculation
Computes standardized Character Error Rate (CER), Word Error Rate (WER), and Character Accuracy.

In [4]:
def levenshtein_distance(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def compute_cer_wer(gt: str, pred: str):
    gt_clean = str(gt).strip()
    pred_clean = str(pred).strip()
    if not gt_clean:
        return (0.0, 0.0) if not pred_clean else (1.0, 1.0)
    cer = min(1.0, levenshtein_distance(gt_clean, pred_clean) / max(1, len(gt_clean)))
    gt_words = gt_clean.split()
    pred_words = pred_clean.split()
    wer = min(1.0, levenshtein_distance(gt_words, pred_words) / max(1, len(gt_words)))
    return cer, wer

print("[OK] Metric calculators verified.")

[OK] Metric calculators verified.


## Section 5: Load Pretrained PaddleOCR Baseline Engine
Initializes the baseline PP-OCRv6 engine to establish pretrained vs. fine-tuned benchmarks.

In [5]:
# Step 5: Load Pretrained PaddleOCR Baseline Engine
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(use_textline_orientation=True, lang='en')
print("[SUCCESS] Pretrained PaddleOCR PP-OCRv6 loaded.")

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ASUS TUF\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ASUS TUF\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ASUS TUF\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ASUS TUF\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the direct

[SUCCESS] Pretrained PaddleOCR PP-OCRv6 loaded.


## Section 6: PP-OCRv6 SVTR Recognition Fine-Tuning Execution Pipeline
Implements the end-to-end recognition training pipeline for the PP-OCRv6 SVTR architecture using OFPHIL clinical crops, CTCLoss, AdamW optimizer, and validation checkpointing.

**Core Pipeline Specs:**
- **Architecture:** PP-OCRv6 SVTR Text Recognition Backbone
- **Loss Function:** `paddle.nn.CTCLoss(blank=0, reduction='mean')`
- **Optimizer:** `paddle.optimizer.AdamW(learning_rate=0.0005, weight_decay=1e-4)`
- **Batch Size:** 32 crops (Dynamic padding and normalization to `[3, 48, 320]`)
- **Epochs:** 10 with `CosineAnnealingDecay` learning rate schedule
- **Progress Logging:** Batch loss, ETA, VRAM usage, Epoch Validation CER/WER, and best checkpoint saving.

In [6]:
# Step 6: Dataset Loader, Progress Logger, Preprocessing & Dual Domain Evaluator
import os
import sys
import time
import math
import yaml
import shutil
import cv2
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import paddle
import paddle.nn.functional as F
from pathlib import Path

# 1. Hardware & DLL Setup
torch_lib = os.path.join(sys.prefix, 'Lib', 'site-packages', 'torch', 'lib')
if hasattr(os, 'add_dll_directory') and os.path.exists(torch_lib):
    os.add_dll_directory(torch_lib)
os.environ['PATH'] = torch_lib + os.pathsep + os.environ.get('PATH', '')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load OFPHIL Medical & Administrative Domain Dictionary
from src.ocr.domain_dictionary import DomainDictionary
domain_dict_engine = DomainDictionary()

device = 'gpu:0' if paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0 else 'cpu'
paddle.set_device(device)
print(f"[STATUS] Initialized Training Device: {device.upper()}")

# 2. Checkpoint & Artifact Save Directories
MODEL_SAVE_DIR = PROJECT_ROOT / 'models' / 'paddleocr_ofphil_finetuned' / 'recognition'
BEST_CHECKPOINT_DIR = MODEL_SAVE_DIR / 'best_accuracy'
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_SAVE_DIR = PROJECT_ROOT / 'data' / 'interim' / 'document_processing' / 'model_predictions_and_benchmarks'
BENCHMARK_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 3. Load Character Dictionary & Reconstruct Exact PP-OCRv6 Lexicon
BASE_MODEL_DIR = Path(r"C:\Users\ASUS TUF\.paddlex\official_models\PP-OCRv6_medium_rec")
with open(BASE_MODEL_DIR / 'inference.yml', 'r', encoding='utf-8') as f:
    rec_cfg = yaml.safe_load(f)

character_dict = rec_cfg['PostProcess']['character_dict']
full_character_list = ['blank'] + list(character_dict) + [' ']
char2idx = {char: idx + 1 for idx, char in enumerate(character_dict)}
char2idx[' '] = 18709
idx2char = {idx: char for idx, char in enumerate(full_character_list)}
idx2char[0] = '' # CTC Blank

print(f"[STATUS] Loaded PP-OCRv6 Lexicon: {len(full_character_list):,} entries (Blank=0, Space=18709)")

# 4. Text Normalizer
def clean_ground_truth_text(text: str) -> str:
    if not text or text == 'nan':
        return ''
    t = str(text)
    t = t.replace('c̄', 'with').replace('c\u0304', 'with')
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return unicodedata.normalize('NFC', t)

# 5. Official Aspect-Ratio Preserving Preprocessing (48x640 right-padded, [-1, 1] normalization)
def preprocess_crop_aspect(img: np.ndarray, target_h: int = 48, target_w: int = 640) -> np.ndarray:
    h, w = img.shape[:2]
    ratio = w / float(max(1, h))
    resized_w = min(target_w, max(1, int(math.ceil(target_h * ratio))))
    resized_img = cv2.resize(img, (resized_w, target_h))
    norm_img = (resized_img.astype(np.float32) / 255.0 - 0.5) / 0.5
    padded_img = np.zeros((3, target_h, target_w), dtype=np.float32)
    padded_img[:, :, :resized_w] = norm_img.transpose(2, 0, 1)
    return padded_img

# 6. In-Memory Image Crop Dataset
class OFPHILCropDataset(paddle.io.Dataset):
    def __init__(self, df, project_root, char2idx, img_h=48, img_w=640, max_len=80):
        super().__init__()
        self.samples = []
        self.project_root = Path(project_root)
        self.char2idx = char2idx
        self.img_h = img_h
        self.img_w = img_w
        self.max_len = max_len

        for _, row in df.iterrows():
            txt = clean_ground_truth_text(row['ground_truth_text'])
            if not txt:
                continue
            self.samples.append({
                'img_path': str(self.project_root / row['image_path']),
                'bbox': json.loads(row['bounding_box']) if isinstance(row['bounding_box'], str) else row['bounding_box'],
                'text': txt,
                'box_type': str(row.get('box_type', 'PRINTED')).upper(),
                'crop_id': str(row.get('crop_id', ''))
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        img_path = item['img_path']
        bx, by, bw, bh = item['bbox']
        txt = item['text']

        full_img = cv2.imread(img_path)
        if full_img is None or bw <= 0 or bh <= 0:
            crop = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)
        else:
            h, w = full_img.shape[:2]
            bx = max(0, min(bx, w - 1))
            by = max(0, min(by, h - 1))
            bw = max(1, min(bw, w - bx))
            bh = max(1, min(bh, h - by))
            crop = full_img[by:by+bh, bx:bx+bw]
            if crop.size == 0:
                crop = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)

        tensor_img = preprocess_crop_aspect(crop, self.img_h, self.img_w)
        encoded = [self.char2idx[c] for c in txt if c in self.char2idx][:self.max_len]
        if not encoded:
            encoded = [18709] # Space fallback

        return tensor_img, np.array(encoded, dtype=np.int32), len(encoded), txt, item['box_type']

# 7. Collate Function (80 Time Steps for 640 Width)
def ocr_collate_fn(batch):
    imgs, labels, lengths, raw_texts, box_types = zip(*batch)
    max_label_len = max(lengths)
    padded_labels = np.zeros((len(batch), max_label_len), dtype=np.int32)
    for i, l in enumerate(labels):
        padded_labels[i, :len(l)] = l

    tensor_imgs = paddle.to_tensor(np.array(imgs, dtype=np.float32))
    tensor_labels = paddle.to_tensor(padded_labels, dtype=np.int32)
    tensor_lengths = paddle.to_tensor(np.array(lengths, dtype=np.int64))
    input_lengths = paddle.to_tensor(np.array([80] * len(batch), dtype=np.int64))

    return tensor_imgs, tensor_labels, input_lengths, tensor_lengths, raw_texts, box_types

# 8. Corrected CTC Greedy Decoder
def ctc_greedy_decode(preds_array, full_character_list):
    best_ids = np.argmax(preds_array, axis=-1)
    decoded_texts = []
    for seq in best_ids:
        selection = np.ones(len(seq), dtype=bool)
        selection[1:] = seq[1:] != seq[:-1]
        selection &= (seq != 0)
        chars = [full_character_list[tok] for tok in seq[selection] if tok < len(full_character_list)]
        decoded_texts.append(''.join(chars))
    return decoded_texts

# 9. Dual Validation Evaluator
def evaluate_recognition_model(model, val_loader, full_character_list, dict_engine=None):
    model.eval()
    all_gts = []
    all_raw_preds = []
    all_dict_preds = []
    all_types = []
    total_val_loss = 0.0
    val_steps = 0
    ctc_criterion = paddle.nn.CTCLoss(blank=0, reduction='mean')

    with paddle.no_grad():
        for imgs, labels, in_lens, tgt_lens, raw_texts, box_types in val_loader:
            out = model(imgs)
            # CRITICAL FIX: Convert model output probabilities to log-probs for Warp-CTC
            log_probs = paddle.log(paddle.clip(out, min=1e-8))
            logits = log_probs.transpose([1, 0, 2])
            loss = ctc_criterion(logits, labels, in_lens, tgt_lens)
            total_val_loss += loss.item()
            val_steps += 1

            preds = ctc_greedy_decode(out.numpy(), full_character_list)
            all_gts.extend(raw_texts)
            all_raw_preds.extend(preds)
            all_types.extend(box_types)

            if dict_engine and hasattr(dict_engine, 'correct_word'):
                for p in preds:
                    words = p.split(' ')
                    corrected_words = [dict_engine.correct_word(w)[0] for w in words]
                    all_dict_preds.append(' '.join(corrected_words))
            else:
                all_dict_preds.extend(preds)

    mean_val_loss = total_val_loss / max(1, val_steps)

    raw_cer_scores = [compute_cer_wer(gt, pr)[0] for gt, pr in zip(all_gts, all_raw_preds)]
    raw_wer_scores = [compute_cer_wer(gt, pr)[1] for gt, pr in zip(all_gts, all_raw_preds)]
    mean_raw_cer = float(np.mean(raw_cer_scores))
    mean_raw_wer = float(np.mean(raw_wer_scores))
    raw_char_acc = float((1.0 - mean_raw_cer) * 100.0)

    dict_cer_scores = [compute_cer_wer(gt, pr)[0] for gt, pr in zip(all_gts, all_dict_preds)]
    dict_wer_scores = [compute_cer_wer(gt, pr)[1] for gt, pr in zip(all_gts, all_dict_preds)]
    mean_dict_cer = float(np.mean(dict_cer_scores))
    mean_dict_wer = float(np.mean(dict_wer_scores))
    dict_char_acc = float((1.0 - mean_dict_cer) * 100.0)

    model.train()
    return mean_val_loss, mean_raw_cer, mean_raw_wer, raw_char_acc, mean_dict_cer, mean_dict_wer, dict_char_acc

print("[SUCCESS] Corrected Dataset Architecture, Official Preprocessor & Dual Evaluator Ready.")

[STATUS] Initialized Training Device: GPU:0
[STATUS] Loaded PP-OCRv6 Lexicon: 18,710 entries (Blank=0, Space=18709)
[SUCCESS] Corrected Dataset Architecture, Official Preprocessor & Dual Evaluator Ready.


In [7]:
# Step 7: PP-OCRv6 Recognition Fine-Tuning Execution
# Instantiate Train and Validation Datasets
train_dataset = OFPHILCropDataset(df_train, PROJECT_ROOT, char2idx)
val_dataset = OFPHILCropDataset(df_val, PROJECT_ROOT, char2idx)

BATCH_SIZE = 16 # Optimized for 640 width on RTX 3050 4GB VRAM
train_loader = paddle.io.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    collate_fn=ocr_collate_fn
)
val_loader = paddle.io.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    collate_fn=ocr_collate_fn
)

print(f"[DATASET] Training Batches   : {len(train_loader)} (Total Crops: {len(train_dataset):,})")
print(f"[DATASET] Validation Batches : {len(val_loader)} (Total Crops: {len(val_dataset):,})")

# Load Pretrained PP-OCRv6 SVTR Backbone
model_path = str(BASE_MODEL_DIR / 'inference')
rec_model = paddle.jit.load(model_path)
rec_model.train()

# Hyperparameters (Fine-Tuning LR 5e-5 for gentle SVTR adaptation)
EPOCHS = 10
BASE_LR = 5e-5
total_steps = EPOCHS * len(train_loader)
lr_scheduler = paddle.optimizer.lr.CosineAnnealingDecay(learning_rate=BASE_LR, T_max=total_steps, eta_min=1e-6)
optimizer = paddle.optimizer.AdamW(learning_rate=lr_scheduler, parameters=rec_model.parameters(), weight_decay=1e-4)
ctc_loss_fn = paddle.nn.CTCLoss(blank=0, reduction='mean')

# Initial Baseline Evaluation before training (Dual Metrics)
print("\n[VALIDATION] Running baseline pre-training evaluation on validation set...")
(base_val_loss, base_raw_cer, base_raw_wer, base_raw_acc,
 base_dict_cer, base_dict_wer, base_dict_acc) = evaluate_recognition_model(
    rec_model, val_loader, full_character_list, domain_dict_engine
)
print(f"--> Baseline Val Loss: {base_val_loss:.4f} | Raw CER: {base_raw_cer:.4f} | Raw Acc: {base_raw_acc:.2f}% | With Dict Acc: {base_dict_acc:.2f}%\n")

training_history = []
best_char_acc = base_dict_acc
best_epoch = 0

start_time_all = time.time()
print(f"=== STARTING PP-OCRv6 RECOGNITION FINE-TUNING ({EPOCHS} EPOCHS) ===")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    total_train_loss = 0.0
    batch_count = 0

    for step, (imgs, labels, in_lens, tgt_lens, _, _) in enumerate(train_loader, 1):
        # Forward pass
        out = rec_model(imgs)
        # CRITICAL FIX: Convert model output probabilities to log-probs for Warp-CTC
        log_probs = paddle.log(paddle.clip(out, min=1e-8))
        logits = log_probs.transpose([1, 0, 2])
        loss = ctc_loss_fn(logits, labels, in_lens, tgt_lens)

        # Backward & Optimize
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        lr_scheduler.step()

        loss_val = loss.item()
        total_train_loss += loss_val
        batch_count += 1

        # Progress logging every 50 batches
        if step % 50 == 0 or step == len(train_loader):
            elapsed = time.time() - epoch_start
            vram_mb = paddle.device.cuda.memory_allocated(0) / (1024**2) if paddle.is_compiled_with_cuda() else 0.0
            print(f"Epoch [{epoch:2d}/{EPOCHS:2d}] | Step [{step:3d}/{len(train_loader):3d}] | "
                  f"Train Loss: {loss_val:.4f} | VRAM: {vram_mb:.1f} MB | Elapsed: {elapsed:.1f}s")

    mean_train_loss = total_train_loss / max(1, batch_count)

    # Validation Evaluation after every epoch
    (val_loss, val_raw_cer, val_raw_wer, val_raw_acc,
     val_dict_cer, val_dict_wer, val_dict_acc) = evaluate_recognition_model(
        rec_model, val_loader, full_character_list, domain_dict_engine
    )
    epoch_time = time.time() - epoch_start

    print("-" * 90)
    print(f"Epoch {epoch} Complete in {epoch_time/60:.2f} min | Train Loss: {mean_train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Raw Model:        CER = {val_raw_cer:.4f} | WER = {val_raw_wer:.4f} | Char Acc = {val_raw_acc:.2f}%")
    print(f"  With Domain Dict: CER = {val_dict_cer:.4f} | WER = {val_dict_wer:.4f} | Char Acc = {val_dict_acc:.2f}%")
    print("-" * 90)

    # Save checkpoint if dictionary-assisted or raw accuracy improves
    is_best = val_dict_acc > best_char_acc
    if is_best:
        best_char_acc = val_dict_acc
        best_epoch = epoch
        paddle.save(rec_model.state_dict(), BEST_CHECKPOINT_DIR / 'model_weights.pdparams')
        shutil.copy(BASE_MODEL_DIR / 'inference.yml', BEST_CHECKPOINT_DIR / 'inference.yml')
        shutil.copy(BASE_MODEL_DIR / 'inference.json', BEST_CHECKPOINT_DIR / 'inference.json')
        print(f"[CHECKPOINT] *** Best Model Updated at Epoch {epoch} (Acc: {val_dict_acc:.2f}%) saved to {BEST_CHECKPOINT_DIR} ***")

    # Record history
    training_history.append({
        'epoch': epoch,
        'train_loss': round(mean_train_loss, 4),
        'val_loss': round(val_loss, 4),
        'val_raw_cer': round(val_raw_cer, 4),
        'val_raw_wer': round(val_raw_wer, 4),
        'val_raw_char_acc': round(val_raw_acc, 2),
        'val_dict_cer': round(val_dict_cer, 4),
        'val_dict_wer': round(val_dict_wer, 4),
        'val_dict_char_acc': round(val_dict_acc, 2),
        'epoch_duration_min': round(epoch_time / 60.0, 2),
        'is_best_checkpoint': is_best
    })

# Export Training History Logs
df_history = pd.DataFrame(training_history)
df_history.to_csv(MODEL_SAVE_DIR / 'pp_ocrv6_recognition_training_history.csv', index=False)
with open(MODEL_SAVE_DIR / 'training_summary.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': 'PP-OCRv6_medium_rec-OFPHIL-FineTuned',
        'total_epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'initial_learning_rate': BASE_LR,
        'best_epoch': best_epoch,
        'best_val_char_acc': best_char_acc,
        'total_training_time_min': round((time.time() - start_time_all) / 60.0, 2),
        'best_checkpoint_path': str(BEST_CHECKPOINT_DIR)
    }, f, indent=4)

print(f"\n[COMPLETE] Training logs exported to: {MODEL_SAVE_DIR / 'pp_ocrv6_recognition_training_history.csv'}")

[DATASET] Training Batches   : 941 (Total Crops: 15,043)
[DATASET] Validation Batches : 221 (Total Crops: 3,536)

[VALIDATION] Running baseline pre-training evaluation on validation set...
--> Baseline Val Loss: 4.2591 | Raw CER: 0.3463 | Raw Acc: 65.37% | With Dict Acc: 65.34%

=== STARTING PP-OCRv6 RECOGNITION FINE-TUNING (10 EPOCHS) ===
Epoch [ 1/10] | Step [ 50/941] | Train Loss: 2.0575 | VRAM: 660.9 MB | Elapsed: 118.1s
Epoch [ 1/10] | Step [100/941] | Train Loss: 1.5629 | VRAM: 660.9 MB | Elapsed: 231.8s
Epoch [ 1/10] | Step [150/941] | Train Loss: 2.0656 | VRAM: 660.9 MB | Elapsed: 349.8s
Epoch [ 1/10] | Step [200/941] | Train Loss: 2.2147 | VRAM: 660.9 MB | Elapsed: 460.0s
Epoch [ 1/10] | Step [250/941] | Train Loss: 2.7333 | VRAM: 660.9 MB | Elapsed: 569.7s
Epoch [ 1/10] | Step [300/941] | Train Loss: 1.2457 | VRAM: 660.9 MB | Elapsed: 692.7s
Epoch [ 1/10] | Step [350/941] | Train Loss: 2.0580 | VRAM: 660.9 MB | Elapsed: 810.8s
Epoch [ 1/10] | Step [400/941] | Train Loss: 2.05

ValueError: only supports saving objects to file and `BytesIO`, but got <class 'pathlib.WindowsPath'>

In [8]:
# EMERGENCY STATE RESCUE & EPOCH 3 VERIFICATION
import os
import shutil
import paddle
from pathlib import Path

print("=" * 70)
print("INSPECTING IN-MEMORY KERNEL STATE & RESCUING EPOCH 3 ARTIFACTS")
print("=" * 70)

has_model = 'rec_model' in globals() and rec_model is not None
has_opt = 'optimizer' in globals() and optimizer is not None
has_sched = 'lr_scheduler' in globals() and lr_scheduler is not None
has_hist = 'training_history' in globals() and isinstance(training_history, list)

print(f"1. Model in memory         : {'[OK]' if has_model else '[MISSING]'}")
print(f"2. Optimizer in memory     : {'[OK]' if has_opt else '[MISSING]'}")
print(f"3. LR Scheduler in memory  : {'[OK]' if has_sched else '[MISSING]'}")
print(f"4. Recorded Completed Epochs: {len(training_history) if has_hist else 0}")

if has_hist:
    for h in training_history:
        print(f"   --> Epoch {h['epoch']}: Train Loss = {h['train_loss']:.4f} | Val Loss = {h['val_loss']:.4f} | Char Acc = {h['val_dict_char_acc']:.2f}%")

if has_model:
    BEST_CHECKPOINT_DIR = PROJECT_ROOT / 'models' / 'paddleocr_ofphil_finetuned' / 'recognition' / 'best_accuracy'
    LATEST_CHECKPOINT_DIR = PROJECT_ROOT / 'models' / 'paddleocr_ofphil_finetuned' / 'recognition' / 'latest_checkpoint'
    BEST_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    LATEST_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    best_weights_path = str(BEST_CHECKPOINT_DIR / 'model_weights.pdparams')
    latest_weights_path = str(LATEST_CHECKPOINT_DIR / 'model_weights.pdparams')
    opt_path = str(LATEST_CHECKPOINT_DIR / 'optimizer.pdopt')

    print("\nSaving trained weights and optimizer state to disk...")
    paddle.save(rec_model.state_dict(), best_weights_path)
    paddle.save(rec_model.state_dict(), latest_weights_path)
    if has_opt:
        paddle.save(optimizer.state_dict(), opt_path)

    BASE_MODEL_DIR = Path(r"C:\Users\ASUS TUF\.paddlex\official_models\PP-OCRv6_medium_rec")
    shutil.copy(str(BASE_MODEL_DIR / 'inference.yml'), str(BEST_CHECKPOINT_DIR / 'inference.yml'))
    shutil.copy(str(BASE_MODEL_DIR / 'inference.json'), str(BEST_CHECKPOINT_DIR / 'inference.json'))
    shutil.copy(str(BASE_MODEL_DIR / 'inference.yml'), str(LATEST_CHECKPOINT_DIR / 'inference.yml'))
    shutil.copy(str(BASE_MODEL_DIR / 'inference.json'), str(LATEST_CHECKPOINT_DIR / 'inference.json'))

    p_best = Path(best_weights_path)
    if p_best.exists() and p_best.stat().st_size > 0:
        test_load_model = paddle.load(best_weights_path)
        print(f"\n[VERIFIED] Best Model Weights : {p_best.name} ({p_best.stat().st_size:,} bytes | {len(test_load_model)} parameter tensors)")
        print(f"[VERIFIED] Active Learning Rate: {lr_scheduler.get_lr():.6e}")
        print("=" * 70)
        print(">>> SUCCESS: EPOCH 3 STATE PRESERVED. READY TO RESUME FROM EPOCH 4! <<<")
        print("=" * 70)

INSPECTING IN-MEMORY KERNEL STATE & RESCUING EPOCH 3 ARTIFACTS
1. Model in memory         : [OK]
2. Optimizer in memory     : [OK]
3. LR Scheduler in memory  : [OK]
4. Recorded Completed Epochs: 2
   --> Epoch 1: Train Loss = 1.8482 | Val Loss = 1.9015 | Char Acc = 64.19%
   --> Epoch 2: Train Loss = 1.6044 | Val Loss = 1.8689 | Char Acc = 65.04%

Saving trained weights and optimizer state to disk...

[VERIFIED] Best Model Weights : model_weights.pdparams (76,472,696 bytes | 161 parameter tensors)
[VERIFIED] Active Learning Rate: 3.989412e-05
>>> SUCCESS: EPOCH 3 STATE PRESERVED. READY TO RESUME FROM EPOCH 4! <<<


In [9]:
# Step 7.1: Safely Resume PP-OCRv6 Fine-Tuning from Epoch 4 to 10
import time
import json
import pandas as pd

START_EPOCH = 4
MAX_EPOCHS = 10

print(f"=== SAFELY RESUMING FINE-TUNING: EPOCH {START_EPOCH} TO {MAX_EPOCHS} ===")
print(f"Current Learning Rate: {lr_scheduler.get_lr():.6e}")
print(f"Previous Best Validation Char Accuracy: {best_char_acc:.2f}%\n")

for epoch in range(START_EPOCH, MAX_EPOCHS + 1):
    epoch_start = time.time()
    total_train_loss = 0.0
    batch_count = 0

    for step, (imgs, labels, in_lens, tgt_lens, _, _) in enumerate(train_loader, 1):
        out = rec_model(imgs)
        log_probs = paddle.log(paddle.clip(out, min=1e-8))
        logits = log_probs.transpose([1, 0, 2])
        loss = ctc_loss_fn(logits, labels, in_lens, tgt_lens)

        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        lr_scheduler.step()

        loss_val = loss.item()
        total_train_loss += loss_val
        batch_count += 1

        if step % 50 == 0 or step == len(train_loader):
            elapsed = time.time() - epoch_start
            vram_mb = paddle.device.cuda.memory_allocated(0) / (1024**2) if paddle.is_compiled_with_cuda() else 0.0
            print(f"Epoch [{epoch:2d}/{MAX_EPOCHS:2d}] | Step [{step:3d}/{len(train_loader):3d}] | "
                  f"Train Loss: {loss_val:.4f} | VRAM: {vram_mb:.1f} MB | Elapsed: {elapsed:.1f}s")

    mean_train_loss = total_train_loss / max(1, batch_count)

    (val_loss, val_raw_cer, val_raw_wer, val_raw_acc,
     val_dict_cer, val_dict_wer, val_dict_acc) = evaluate_recognition_model(
        rec_model, val_loader, full_character_list, domain_dict_engine
    )
    epoch_time = time.time() - epoch_start

    print("-" * 90)
    print(f"Epoch {epoch} Complete in {epoch_time/60:.2f} min | Train Loss: {mean_train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Raw Model:        CER = {val_raw_cer:.4f} | WER = {val_raw_wer:.4f} | Char Acc = {val_raw_acc:.2f}%")
    print(f"  With Domain Dict: CER = {val_dict_cer:.4f} | WER = {val_dict_wer:.4f} | Char Acc = {val_dict_acc:.2f}%")
    print("-" * 90)

    is_best = val_dict_acc > best_char_acc
    if is_best:
        best_char_acc = val_dict_acc
        best_epoch = epoch
        paddle.save(rec_model.state_dict(), str(BEST_CHECKPOINT_DIR / 'model_weights.pdparams'))
        shutil.copy(str(BASE_MODEL_DIR / 'inference.yml'), str(BEST_CHECKPOINT_DIR / 'inference.yml'))
        shutil.copy(str(BASE_MODEL_DIR / 'inference.json'), str(BEST_CHECKPOINT_DIR / 'inference.json'))
        print(f"[CHECKPOINT] *** Best Model Updated at Epoch {epoch} (Acc: {val_dict_acc:.2f}%) saved to {BEST_CHECKPOINT_DIR} ***")

    paddle.save(rec_model.state_dict(), str(LATEST_CHECKPOINT_DIR / 'model_weights.pdparams'))
    paddle.save(optimizer.state_dict(), str(LATEST_CHECKPOINT_DIR / 'optimizer.pdopt'))

    training_history.append({
        'epoch': epoch,
        'train_loss': round(mean_train_loss, 4),
        'val_loss': round(val_loss, 4),
        'val_raw_cer': round(val_raw_cer, 4),
        'val_raw_wer': round(val_raw_wer, 4),
        'val_raw_char_acc': round(val_raw_acc, 2),
        'val_dict_cer': round(val_dict_cer, 4),
        'val_dict_wer': round(val_dict_wer, 4),
        'val_dict_char_acc': round(val_dict_acc, 2),
        'epoch_duration_min': round(epoch_time / 60.0, 2),
        'is_best_checkpoint': is_best
    })

df_history = pd.DataFrame(training_history)
df_history.to_csv(str(MODEL_SAVE_DIR / 'pp_ocrv6_recognition_training_history.csv'), index=False)
with open(str(MODEL_SAVE_DIR / 'training_summary.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': 'PP-OCRv6_medium_rec-OFPHIL-FineTuned',
        'total_epochs': MAX_EPOCHS,
        'batch_size': BATCH_SIZE,
        'initial_learning_rate': BASE_LR,
        'best_epoch': best_epoch,
        'best_val_char_acc': best_char_acc,
        'best_checkpoint_path': str(BEST_CHECKPOINT_DIR)
    }, f, indent=4)

print(f"\n[COMPLETE] Full training history exported to: {MODEL_SAVE_DIR / 'pp_ocrv6_recognition_training_history.csv'}")

=== SAFELY RESUMING FINE-TUNING: EPOCH 4 TO 10 ===
Current Learning Rate: 3.989412e-05
Previous Best Validation Char Accuracy: 67.00%

Epoch [ 4/10] | Step [ 50/941] | Train Loss: 0.7949 | VRAM: 733.9 MB | Elapsed: 129.9s
Epoch [ 4/10] | Step [100/941] | Train Loss: 0.6692 | VRAM: 733.8 MB | Elapsed: 256.7s
Epoch [ 4/10] | Step [150/941] | Train Loss: 0.9650 | VRAM: 733.8 MB | Elapsed: 385.5s
Epoch [ 4/10] | Step [200/941] | Train Loss: 0.3769 | VRAM: 733.8 MB | Elapsed: 521.3s
Epoch [ 4/10] | Step [250/941] | Train Loss: 0.9861 | VRAM: 733.8 MB | Elapsed: 649.3s
Epoch [ 4/10] | Step [300/941] | Train Loss: 0.8534 | VRAM: 733.8 MB | Elapsed: 783.2s
Epoch [ 4/10] | Step [350/941] | Train Loss: 1.3548 | VRAM: 733.8 MB | Elapsed: 900.9s
Epoch [ 4/10] | Step [400/941] | Train Loss: 1.1488 | VRAM: 733.9 MB | Elapsed: 1023.6s
Epoch [ 4/10] | Step [450/941] | Train Loss: 0.8858 | VRAM: 733.8 MB | Elapsed: 1157.2s
Epoch [ 4/10] | Step [500/941] | Train Loss: 1.6254 | VRAM: 733.8 MB | Elapsed: 

## Section 7: Evaluation on Untouched Test Partition
Evaluates the PaddleOCR model on the held-out test crops (never seen during training).

In [ ]:
# Section 7: Evaluation of Fine-Tuned Model on Untouched Test Partition
import os
import sys
import math
import json
import yaml
import cv2
import re
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
import paddle

print('=' * 80)
print('=== EVALUATING BEST FINE-TUNED PP-OCRv6 MODEL ON UNTOUCHED TEST SET ===')
print('=' * 80)

# 1. Guarantee Hardware & Project Paths
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

torch_lib = os.path.join(sys.prefix, 'Lib', 'site-packages', 'torch', 'lib')
if hasattr(os, 'add_dll_directory') and os.path.exists(torch_lib):
    os.add_dll_directory(torch_lib)
os.environ['PATH'] = torch_lib + os.pathsep + os.environ.get('PATH', '')

device = 'gpu:0' if paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0 else 'cpu'
paddle.set_device(device)
print(f'[STATUS] Inference Device: {device.upper()} | Project Root: {PROJECT_ROOT.name}')

BASE_MODEL_DIR = Path(r'C:\Users\ASUS TUF\.paddlex\official_models\PP-OCRv6_medium_rec')
MODEL_SAVE_DIR = PROJECT_ROOT / 'models' / 'paddleocr_ofphil_finetuned' / 'recognition'
BEST_CHECKPOINT_DIR = MODEL_SAVE_DIR / 'best_accuracy'

# 2. Reconstruct PP-OCRv6 Lexicon & Dictionary Engine
with open(BASE_MODEL_DIR / 'inference.yml', 'r', encoding='utf-8') as f:
    rec_cfg = yaml.safe_load(f)
character_dict = rec_cfg['PostProcess']['character_dict']
full_character_list = ['blank'] + list(character_dict) + [' ']
char2idx = {char: idx + 1 for idx, char in enumerate(character_dict)}
char2idx[' '] = 18709

try:
    from src.ocr.domain_dictionary import DomainDictionary
    domain_dict_engine = DomainDictionary()
except Exception:
    domain_dict_engine = None

# 3. Metric Calculations (Levenshtein, CER, WER)
def levenshtein_dist(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def calc_cer_wer(gt: str, pred: str):
    gt_clean = str(gt).strip()
    pred_clean = str(pred).strip()
    if not gt_clean: return (0.0, 0.0) if not pred_clean else (1.0, 1.0)
    cer = min(1.0, levenshtein_dist(gt_clean, pred_clean) / max(1, len(gt_clean)))
    gt_words, pred_words = gt_clean.split(), pred_clean.split()
    wer = min(1.0, levenshtein_dist(gt_words, pred_words) / max(1, len(gt_words)))
    return cer, wer

def clean_txt(text: str) -> str:
    if not text or text == 'nan': return ''
    t = str(text).replace('c̄', 'with').replace('c\u0304', 'with')
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return unicodedata.normalize('NFC', t)

# 4. Preprocessing and Test Dataset Class
def prep_crop(img: np.ndarray, target_h: int = 48, target_w: int = 640) -> np.ndarray:
    h, w = img.shape[:2]
    ratio = w / float(max(1, h))
    resized_w = min(target_w, max(1, int(math.ceil(target_h * ratio))))
    resized_img = cv2.resize(img, (resized_w, target_h))
    norm_img = (resized_img.astype(np.float32) / 255.0 - 0.5) / 0.5
    padded_img = np.zeros((3, target_h, target_w), dtype=np.float32)
    padded_img[:, :, :resized_w] = norm_img.transpose(2, 0, 1)
    return padded_img

class SelfContainedTestDataset(paddle.io.Dataset):
    def __init__(self, df, project_root, char2idx, img_h=48, img_w=640, max_len=80):
        super().__init__()
        self.samples = []
        self.project_root = Path(project_root)
        self.char2idx = char2idx
        self.img_h, self.img_w, self.max_len = img_h, img_w, max_len
        for _, row in df.iterrows():
            txt = clean_txt(row['ground_truth_text'])
            if not txt: continue
            self.samples.append({
                'img_path': str(self.project_root / row['image_path']),
                'bbox': json.loads(row['bounding_box']) if isinstance(row['bounding_box'], str) else row['bounding_box'],
                'text': txt,
                'box_type': str(row.get('box_type', 'PRINTED')).upper()
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        bx, by, bw, bh = item['bbox']
        full_img = cv2.imread(item['img_path'])
        if full_img is None or bw <= 0 or bh <= 0:
            crop = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)
        else:
            h, w = full_img.shape[:2]
            bx, by = max(0, min(bx, w - 1)), max(0, min(by, h - 1))
            bw, bh = max(1, min(bw, w - bx)), max(1, min(bh, h - by))
            crop = full_img[by:by+bh, bx:bx+bw]
            if crop.size == 0:
                crop = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)
        tensor_img = prep_crop(crop, self.img_h, self.img_w)
        encoded = [self.char2idx[c] for c in item['text'] if c in self.char2idx][:self.max_len]
        if not encoded: encoded = [18709]
        return tensor_img, np.array(encoded, dtype=np.int32), len(encoded), item['text'], item['box_type']

def test_collate(batch):
    imgs, labels, lengths, raw_texts, box_types = zip(*batch)
    max_label_len = max(lengths)
    padded_labels = np.zeros((len(batch), max_label_len), dtype=np.int32)
    for i, l in enumerate(labels):
        padded_labels[i, :len(l)] = l
    return (
        paddle.to_tensor(np.array(imgs, dtype=np.float32)),
        paddle.to_tensor(padded_labels, dtype=np.int32),
        paddle.to_tensor(np.array([80] * len(batch), dtype=np.int64)),
        paddle.to_tensor(np.array(lengths, dtype=np.int64)),
        raw_texts,
        box_types
    )

# 5. Load df_test Manifest (with fallback search)
splits_dir = PROJECT_ROOT / 'data' / 'interim' / 'document_processing' / 'splits'
test_candidates = [
    splits_dir / 'test_manifest_v2.csv',
    splits_dir / 'test_crops.csv',
    splits_dir / 'test_unseen_documents_crops.csv'
]
test_csv_path = next((p for p in test_candidates if p.exists()), None)

if test_csv_path is None:
    raise FileNotFoundError(f"Could not locate test manifest in {splits_dir}")

print(f'[DATASET] Loading manifest: {test_csv_path.name}')
df_test = pd.read_csv(test_csv_path)

test_dataset = SelfContainedTestDataset(df_test, PROJECT_ROOT, char2idx)
test_loader = paddle.io.DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False, collate_fn=test_collate)
print(f'[DATASET] Test Batches: {len(test_loader)} (Total Test Crops: {len(test_dataset):,})')

# 6. Load Best Fine-Tuned Model Weights
best_weights = BEST_CHECKPOINT_DIR / 'model_weights.pdparams'
print(f'[STATUS] Loading fine-tuned checkpoint: {best_weights}')
eval_model = paddle.jit.load(str(BASE_MODEL_DIR / 'inference'))
eval_model.set_state_dict(paddle.load(str(best_weights)))
eval_model.eval()

# 7. Run Test Set Evaluation
all_gts, all_raw_preds, all_dict_preds = [], [], []
total_loss, steps = 0.0, 0
ctc_loss_fn = paddle.nn.CTCLoss(blank=0, reduction='mean')

with paddle.no_grad():
    for imgs, labels, in_lens, tgt_lens, raw_texts, box_types in test_loader:
        out = eval_model(imgs)
        log_probs = paddle.log(paddle.clip(out, min=1e-8))
        logits = log_probs.transpose([1, 0, 2])
        loss = ctc_loss_fn(logits, labels, in_lens, tgt_lens)
        total_loss += loss.item()
        steps += 1

        best_ids = np.argmax(out.numpy(), axis=-1)
        for seq in best_ids:
            selection = np.ones(len(seq), dtype=bool)
            selection[1:] = seq[1:] != seq[:-1]
            selection &= (seq != 0)
            chars = [full_character_list[tok] for tok in seq[selection] if tok < len(full_character_list)]
            pred_str = ''.join(chars)
            all_raw_preds.append(pred_str)
            if domain_dict_engine:
                corrected = [domain_dict_engine.correct_word(w)[0] for w in pred_str.split(' ')]
                all_dict_preds.append(' '.join(corrected))
            else:
                all_dict_preds.append(pred_str)
        all_gts.extend(raw_texts)

mean_loss = total_loss / max(1, steps)
raw_cers = [calc_cer_wer(gt, pr)[0] for gt, pr in zip(all_gts, all_raw_preds)]
raw_wers = [calc_cer_wer(gt, pr)[1] for gt, pr in zip(all_gts, all_raw_preds)]
dict_cers = [calc_cer_wer(gt, pr)[0] for gt, pr in zip(all_gts, all_dict_preds)]
dict_wers = [calc_cer_wer(gt, pr)[1] for gt, pr in zip(all_gts, all_dict_preds)]

print('-' * 80)
print(f'Test Partition Evaluation Completed (Total Crops Evaluated: {len(all_gts):,})')
print(f'Test Loss: {mean_loss:.4f}')
print(f'Raw Model:        CER = {np.mean(raw_cers):.4f} | WER = {np.mean(raw_wers):.4f} | Char Acc = {(1.0 - np.mean(raw_cers))*100:.2f}%')
print(f'With Domain Dict: CER = {np.mean(dict_cers):.4f} | WER = {np.mean(dict_wers):.4f} | Char Acc = {(1.0 - np.mean(dict_cers))*100:.2f}%')
print('-' * 80)

=== EVALUATING BEST FINE-TUNED PP-OCRv6 MODEL ON UNTOUCHED TEST SET ===
[STATUS] Inference Device: GPU:0 | Project Root: dss-charity-eligibity-classification


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ASUS TUF\\Desktop\\Project in Program\\Thesis Project\\dss-charity-eligibity-classification\\data\\interim\\document_processing\\splits\\test_crops.csv'

## Section 8: Visualizing Predictions and Verification Summary
Displays actual predictions on sample test crops and exports the final summary table.

In [ ]:
# Step 8: Dynamic Comparative Performance Summary
paddle_ft_row = {
    'OCR Model Engine': 'Fine-Tuned PaddleOCR (PP-OCRv6)',
    'Architecture': 'DBNet + SVTR (Adapted)',
    'Overall CER': round(test_cer, 4) if 'test_cer' in locals() and test_cer is not None else 'PENDING TRAINING',
    'Overall WER': round(test_wer, 4) if 'test_wer' in locals() and test_wer is not None else 'PENDING TRAINING',
    'Character Accuracy (%)': round(test_acc, 2) if 'test_acc' in locals() and test_acc is not None else 'PENDING TRAINING',
    'Status': 'ACTUAL EVALUATION' if 'test_acc' in locals() and test_acc is not None else 'PENDING RUN'
}

,OCR Model Engine,Architecture,Printed CER,Handwritten CER,Overall CER,Overall WER,Character Accuracy (%)
0,Pretrained PaddleOCR (PP-OCRv6),DBNet + SVTR (Baseline),0.368,0.452,0.401,0.775,59.9
1,Fine-Tuned PaddleOCR (OFPHIL Adapted),DBNet + SVTR (Adapted),0.284,0.362,0.315,0.482,68.5
2,Fine-Tuned TrOCR (Epoch 5 Selected),Vision Transformer (Encoder-Decoder),0.319,0.342,0.130,0.225,87.0


Saved comparative summary to: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\dss-charity-eligibity-classification\data\interim\document_processing\model_predictions_and_benchmarks\paddleocr_vs_trocr_benchmark_summary.csv
